# **Data Consolidation for Bus, Subway, and Streetcar Datasets (2023–2025)**

## **1.** Combining **Bus Delay** data for the year 2023, 2024 and 2025; saved as **TTC_Bus_combined.csv**

In [ ]:
import pandas as pd

# ----------------------------
# 1. Load datasets
# ----------------------------
df_2023 = pd.read_excel("/content/sample_data/ttc-bus-delay-data-2023.xlsx")
df_2024 = pd.read_excel("/content/sample_data/ttc-bus-delay-data-2024.xlsx")
df_2025 = pd.read_csv("/content/sample_data/TTC_Bus_Delay_Data_2025.csv")


In [ ]:
df_2023.head(),df_2024.head(),df_2025.head()

(        Date Line   Time     Day                Station  \
 0 2023-01-01   91  02:30  Sunday  WOODBINE AND MORTIMER   
 1 2023-01-01   69  02:34  Sunday         WARDEN STATION   
 2 2023-01-01   35  03:06  Sunday           JANE STATION   
 3 2023-01-01  900  03:14  Sunday        KIPLING STATION   
 4 2023-01-01   85  03:43  Sunday         MEADOWALE LOOP   
 
                     Code  Min Delay  Min Gap Bound  Vehicle Transport_Type  
 0              Diversion         81      111   NaN     8772            Bus  
 1               Security         22       44     S     8407            Bus  
 2  Cleaning - Unsanitary         30       60     N     1051            Bus  
 3               Security         17       17   NaN     3334            Bus  
 4               Security          1        1   NaN     1559            Bus  ,
         Date Line   Time     Day               Station           Code  \
 0 2024-01-01   89  02:08  Monday    KEELE AND GLENLAKE         Vision   
 1 2024-01-01   39  0

#### TTC data for 2023 and 2024 does not explicitly label weather-related delays. Instead, weather impacts are indirectly reflected through categories such as diversions, mechanical failures, collisions, and emergency incidents, so we only chose 'Diversion','Mechanical', 'Collision - TTC',and 'Emergency Services'.


In [ ]:
# ----------------------------
# 2. Define categories to keep
# ----------------------------
target_categories = [
    'Diversion',
    'Mechanical',
    'Collision - TTC',
    'Emergency Services'
]

# ----------------------------
# 3. Filter 2023 and 2024
# ----------------------------
df_2023_filtered = df_2023[df_2023['Code'].isin(target_categories)].copy()
df_2024_filtered = df_2024[df_2024['Code'].isin(target_categories)].copy()


#### TTC Data for 2025 used coded delay categories, which were mapped to standardized categories such as Diversion, Mechanical, Collision, and Emergency Services. Non-relevant codes were excluded to ensure consistency with earlier 2023 and 2024 datasets

In [ ]:
# ----------------------------
# 4. Map 2025 coded values
# ----------------------------
code_mapping_2025 = {
    # Diversion
    'MFDV': 'Mechanical',

    # Mechanical
    'MFESA': 'Mechanical',
    'MFSAN': 'Mechanical',
    'MFUS': 'Mechanical',
    'MFUI': 'Mechanical',
    'TFCNO': 'Mechanical',

    # Emergency Services
    'EFO': 'Emergency Services',
    'EFP': 'Emergency Services',

    # Collision - TTC
    'TFPD': 'Emergency Services',
    'TFPI': 'Emergency Services',

    # Weather Related
    'MFWEA': 'Mechanical'
}

# Remove _id column from 2025 if present
if '_id' in df_2025.columns:
    df_2025 = df_2025.drop(columns=['_id'])

# Map 2025 codes
df_2025['Code'] = df_2025['Code'].map(code_mapping_2025)

# Keep only mapped rows
df_2025_filtered = df_2025[df_2025['Code'].isin(target_categories)].copy()

In [ ]:

# ----------------------------
# 5. Standardize Date columns
# ----------------------------
df_2023_filtered['Date'] = pd.to_datetime(df_2023_filtered['Date'], errors='coerce')
df_2024_filtered['Date'] = pd.to_datetime(df_2024_filtered['Date'], errors='coerce')
df_2025_filtered['Date'] = pd.to_datetime(df_2025_filtered['Date'], errors='coerce')


# ----------------------------
# 6. Keep same columns in same order
# ----------------------------
common_cols = [
    'Date', 'Line', 'Time', 'Day', 'Station',
    'Code', 'Min Delay', 'Min Gap', 'Bound',
    'Vehicle', 'Transport_Type'
]

df_2023_filtered = df_2023_filtered[common_cols]
df_2024_filtered = df_2024_filtered[common_cols]
df_2025_filtered = df_2025_filtered[common_cols]

# ----------------------------
# 7. Combine all three datasets
# ----------------------------
df_all = pd.concat(
    [df_2023_filtered, df_2024_filtered, df_2025_filtered],
    ignore_index=True
)

# Since the subway dataset only contained two dominant categories - Mechanical and Emergency Services
# it was necessary to standardize the categorization across bus and streetcar datasets
# to maintain consistency in the analysis.

merge_mapping = {
    'Diversion': 'Mechanical',
    'Mechanical': 'Mechanical',
    'Collision - TTC': 'Emergency Services',
    'Emergency Services': 'Emergency Services',
}

df_all['Code'] = df_all['Code'].map(merge_mapping)


# ----------------------------
# 8. Save combined dataset
# ----------------------------
df_all.to_csv("TTC_Bus_combined.csv", index=False)

# ----------------------------
# 9. Preview
# ----------------------------
print(df_all.head())
print(df_all.tail())
print("\nShape:", df_all.shape)
print("\nCode counts:")
print(df_all['Code'].value_counts())
print("\nDate sample:")
print(df_all['Date'].head())

        Date Line   Time     Day                Station                Code  \
0 2023-01-01   91  02:30  Sunday  WOODBINE AND MORTIMER          Mechanical   
1 2023-01-01   40  03:47  Sunday        KIPLING STATION  Emergency Services   
2 2023-01-01  336  03:52  Sunday       FINCH AND ALNESS          Mechanical   
3 2023-01-01   52  04:25  Sunday     LAWRENCE AND YONGE  Emergency Services   
4 2023-01-01   36  05:18  Sunday       FINCH AND ALNESS          Mechanical   

   Min Delay  Min Gap Bound  Vehicle Transport_Type  
0         81      111   NaN     8772            Bus  
1          0        0   NaN        0            Bus  
2        138      168   NaN     9220            Bus  
3         30       60     E     3520            Bus  
4        334      344   NaN     3524            Bus  
            Date                    Line  Time        Day  \
99035 2025-12-31              131 NUGGET  0:28  Wednesday   
99036 2025-12-31         116 MORNINGSIDE  0:31  Wednesday   
99037 2025-12-31  

## **2.** Combining **Subway Delay** data for the year 2023, 2024 and 2025; saved as **TTC_Subway_combined.csv**

#### For the 2023, 2024 and 2025 subway dataset, the delay code mapping primarily supported two relevant categories: Mechanical and Emergency Services. Mechanical-related incidents were represented by codes beginning with MU, TU, EU, and PU while emergency or response-related incidents were represented by SU and PU codes. No clear subway code equivalents were identified for Diversion or Collision - TTC, so these categories were not included in the subway-specific analysis.

In [ ]:
# ----------------------------
# 1. Load data
# ----------------------------
df_2023 = pd.read_excel("/content/sample_data/ttc-subway-delay-data-2023.xlsx")
df_2024 = pd.read_excel("/content/sample_data/ttc-subway-delay-data-2024.xlsx")
df_2025 = pd.read_csv("/content/sample_data/TTC_Subway_Delay_Data_2025.csv")

# Remove _id if exists
if '_id' in df_2025.columns:
    df_2025 = df_2025.drop(columns=['_id'])

# ----------------------------
# 2. Mapping function (UPDATED)
# ----------------------------
def map_subway_category(code):
    code = str(code)

    # Mechanical (equipment, track, electrical)
    if code.startswith(('MU', 'TU', 'EU','PU')):
        return 'Mechanical'

    # Emergency / external / passenger
    elif code.startswith(('SU', 'PU')):
        return 'Emergency Services'

    else:
        return None

# ----------------------------
# 3. Apply mapping
# ----------------------------
for df in [df_2023, df_2024, df_2025]:
    df['Mapped_Code'] = df['Code'].apply(map_subway_category)

# ----------------------------
# 4. Keep only relevant rows
# ----------------------------
df_2023_f = df_2023[df_2023['Mapped_Code'].notna()].copy()
df_2024_f = df_2024[df_2024['Mapped_Code'].notna()].copy()
df_2025_f = df_2025[df_2025['Mapped_Code'].notna()].copy()

# ----------------------------
# 5. Replace Code column
# ----------------------------
for df in [df_2023_f, df_2024_f, df_2025_f]:
    df['Code'] = df['Mapped_Code']
    df.drop(columns=['Mapped_Code'], inplace=True)

# ----------------------------
# 6. Standardize Date
# ----------------------------
for df in [df_2023_f, df_2024_f, df_2025_f]:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# ----------------------------
# 7. Align columns
# ----------------------------
common_cols = [
    'Date', 'Line', 'Time', 'Day', 'Station',
    'Code', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle','Transport_Type'
]

df_2023_f = df_2023_f[common_cols]
df_2024_f = df_2024_f[common_cols]
df_2025_f = df_2025_f[common_cols]

# ----------------------------
# 8. Combine all datasets
# ----------------------------
df_all = pd.concat(
    [df_2023_f, df_2024_f, df_2025_f],
    ignore_index=True
)

# ----------------------------
# 9. Save final dataset
# ----------------------------
df_all.to_csv("TTC_Subway_combined.csv", index=False)

# ----------------------------
# 10. Check results
# ----------------------------
print(df_all.head())
print("\nShape:", df_all.shape)
print("\nCode counts:")
print(df_all['Code'].value_counts())

        Date Line   Time     Day           Station                Code  \
0 2023-01-01   YU  02:22  Sunday    MUSEUM STATION          Mechanical   
1 2023-01-01   BD  02:30  Sunday   KIPLING STATION          Mechanical   
2 2023-01-01   BD  02:33  Sunday    WARDEN STATION  Emergency Services   
3 2023-01-01   BD  03:17  Sunday     KEELE STATION          Mechanical   
4 2023-01-01   BD  07:16  Sunday  BATHURST STATION          Mechanical   

   Min Delay  Min Gap Bound  Vehicle Transport_Type  
0          3        9     S     5931         Subway  
1          0        0     E     5341         Subway  
2          0        0     W        0         Subway  
3          0        0   NaN        0         Subway  
4          0        0   NaN        0         Subway  

Shape: (74395, 11)

Code counts:
Code
Mechanical            51632
Emergency Services    22763
Name: count, dtype: int64


## **3.** Combining **Bus Delay** data for the year 2023, 2024 and 2025; saved as **TTC_Streetcar_combined.csv**

In [ ]:
import pandas as pd

# ----------------------------
# 1. Load datasets
# ----------------------------
df_2023 = pd.read_excel("/content/sample_data/ttc-streetcar-delay-data-2023.xlsx")
df_2024 = pd.read_excel("/content/sample_data/ttc-streetcar-delay-data-2024.xlsx")
df_2025 = pd.read_csv("/content/sample_data/TTC_Streetcar_Delay_Data_2025.csv")


#### TTC data for 2023 and 2024 does not explicitly label weather-related delays. Instead, weather impacts are indirectly reflected through categories such as diversions, mechanical failures, collisions, and emergency incidents, so we only chose 'Diversion','Mechanical', 'Collision - TTC',and 'Emergency Services'.

In [ ]:
# ----------------------------
# 2. Define categories to keep
# ----------------------------
target_categories = [
    'Diversion',
    'Mechanical',
    'Collision - TTC Involved',
    'Emergency Services'
]

# ----------------------------
# 3. Filter 2023 and 2024
# ----------------------------
df_2023_filtered = df_2023[df_2023['Code'].isin(target_categories)].copy()
df_2024_filtered = df_2024[df_2024['Code'].isin(target_categories)].copy()


#### TTC Data for 2025 used coded delay categories, which were mapped to standardized categories such as Diversion, Mechanical, Collision, and Emergency Services. Non-relevant codes were excluded to ensure consistency with earlier 2023 and 2024 datasets

In [ ]:
# Remove _id if present
if '_id' in df.columns:
    df = df.drop(columns=['_id'])

streetcar_code_map = {
    # Diversion
    'MTDV': 'Diversion',

    # Weather Related
    'MTWEA': 'Mechanical',

    # Collision - TTC
    'PTPD': 'Collision - TTC Involved',
    'TTPD': 'Collision - TTC Involved',
    'TTPI': 'Collision - TTC Involved',

    # Emergency Services
    'MTIE': 'Emergency Services',
    'MTPOL': 'Emergency Services',
    'MTS': 'Emergency Services',
    'MTUI': 'Emergency Services',
    'MTUIR': 'Emergency Services',
    'TTOI': 'Emergency Services',
    'STAE': 'Emergency Services',
    'STAP': 'Emergency Services',
    'STBT': 'Emergency Services',
    'STDP': 'Emergency Services',
    'STEAS': 'Emergency Services',
    'STSA': 'Emergency Services',
    'STSP': 'Emergency Services',

    # Mechanical
    'ETAC': 'Mechanical',
    'ETAR': 'Mechanical',
    'ETAX': 'Mechanical',
    'ETBO': 'Mechanical',
    'ETCA': 'Mechanical',
    'ETCE': 'Mechanical',
    'ETCH': 'Mechanical',
    'ETCM': 'Mechanical',
    'ETDB': 'Mechanical',
    'ETDO': 'Mechanical',
    'ETDS': 'Mechanical',
    'ETFA': 'Mechanical',
    'ETHV': 'Mechanical',
    'ETLT': 'Mechanical',
    'ETLV': 'Mechanical',
    'ETNEA': 'Mechanical',
    'ETNT': 'Mechanical',
    'ETO': 'Mechanical',
    'ETPI': 'Mechanical',
    'ETRA': 'Mechanical',
    'ETSA': 'Mechanical',
    'ETSE': 'Mechanical',
    'ETTB': 'Mechanical',
    'ETTM': 'Mechanical',
    'ETTR': 'Mechanical',
    'ETVC': 'Mechanical',
    'ETVE': 'Mechanical',
    'ETWA': 'Mechanical',
    'ETWF': 'Mechanical',
    'ETWS': 'Mechanical',
    'PTNTF': 'Mechanical',
    'PTO': 'Mechanical',
    'PTOV': 'Mechanical',
    'PTSE': 'Mechanical',
    'PTSW': 'Mechanical',
    'PTTR': 'Mechanical',
    'PTW': 'Mechanical',
}

target_categories = [
    'Diversion',
    'Mechanical',
    'Collision - TTC Involved',
    'Emergency Services'
]


# Remove _id column from 2025 if present
if '_id' in df_2025.columns:
    df_2025 = df_2025.drop(columns=['_id'])

# Map 2025 codes
df_2025['Code'] = df_2025['Code'].map(code_mapping_2025)

# Keep only mapped rows
df_2025_filtered = df_2025[df_2025['Code'].isin(target_categories)].copy()



In [ ]:
# ----------------------------
# 5. Standardize Date columns
# ----------------------------
df_2023_filtered['Date'] = pd.to_datetime(df_2023_filtered['Date'], errors='coerce')
df_2024_filtered['Date'] = pd.to_datetime(df_2024_filtered['Date'], errors='coerce')
df_2025_filtered['Date'] = pd.to_datetime(df_2025_filtered['Date'], errors='coerce')


# ----------------------------
# 6. Keep same columns in same order
# ----------------------------
common_cols = [
    'Date', 'Line', 'Time', 'Day', 'Station',
    'Code', 'Min Delay', 'Min Gap', 'Bound',
    'Vehicle', 'Transport_Type'
]

df_2023_filtered = df_2023_filtered[common_cols]
df_2024_filtered = df_2024_filtered[common_cols]
df_2025_filtered = df_2025_filtered[common_cols]

# ----------------------------
# 7. Combine all three datasets
# ----------------------------
df_all = pd.concat(
    [df_2023_filtered, df_2024_filtered, df_2025_filtered],
    ignore_index=True
)


# Since the subway dataset only contained two dominant categories - Mechanical and Emergency Services
# it was necessary to standardize the categorization across bus and streetcar datasets
# to maintain consistency in the analysis.

merge_mapping = {
    'Diversion': 'Mechanical',
    'Mechanical': 'Mechanical',
    'Collision - TTC Involved': 'Emergency Services',
    'Emergency Services': 'Emergency Services'
}

df_all['Code'] = df_all['Code'].map(merge_mapping)


# ----------------------------
# 8. Save combined dataset
# ----------------------------
df_all.to_csv("TTC_Streetcar_combined.csv", index=False)

# ----------------------------
# 9. Preview
# ----------------------------
print(df_all.head())
print(df_all.tail())
print("\nShape:", df_all.shape)
print("\nCode counts:")
print(df_all['Code'].value_counts())
print("\nDate sample:")
print(df_all['Date'].head())

        Date Line   Time     Day                 Station                Code  \
0 2023-01-01  505  03:53  Sunday    LANSDOWNE AND DUNDAS  Emergency Services   
1 2023-01-01  506  09:58  Sunday  PARLIAMENT AND ST DAVI  Emergency Services   
2 2023-01-01  510  11:31  Sunday  QUEENSQUAY AND SPADINA  Emergency Services   
3 2023-01-01  501  14:21  Sunday         QUEEN AND PETER  Emergency Services   
4 2023-01-01  510  15:19  Sunday    HARBOUR FRONT TUNNEL  Emergency Services   

   Min Delay  Min Gap Bound  Vehicle Transport_Type  
0          0        0     N     4460      Streetcar  
1         10       19     S     4519      Streetcar  
2         14       28   NaN     4448      Streetcar  
3          8       15     W     4582      Streetcar  
4          0        0     W     4534      Streetcar  
            Date               Line   Time        Day                 Station  \
11166 2025-12-29  503 KINGSTON ROAD  13:27     Monday            BINGHAM LOOP   
11167 2025-12-31  503 KINGSTON RO